# 02 · Tall column, then iron the outer faces with the second arm
Nino's experiment in four parts that can each be run on their own:

1. **Column**: one arm prints a tall cylinder (bed does the motion, as `src/cell/runs/cylinder.py`).
2. **Face tracking, cold and clear**: the second arm follows the outer face at a set height, a few millimetres off it, while the bed turns. Proves the geometry before anything touches.
3. **Ironing pass**: Nino's zones and motifs (line, zigzag, spiral): the nozzle (hot, extruder off) presses the face by each zone's `pressure` and the bed turns the face past it.
4. **Opposite faces**: the arms sit 180° apart, so one face is ironed while the other is printed; then the bed brings the other round.

The zone list is where the technique lives: change `IRON_ZONES` to try a pattern.

Settings are in the first cell. Simulators first (`SIM = True`).

## Settings

In [ ]:
import sys, os, time, math, threading
ROOT = os.path.abspath("..")
sys.path[:0] = [ROOT, os.path.join(ROOT, "src"), os.path.join(ROOT, "ui")]   # not needed after: pip install -e .
import cell.runs.cylinder as cyl
from cell.runs.cylinder import Cell, PARK, SAFE_Z, SIDE, WATCHDOG_S

SIM = True
PRINT_ARM, IRON_ARM = 0, 1          # right prints, left irons (swap to test the other way)

COLUMN = dict(T=PRINT_ARM, R=40, E=4, L_H=0.6, HEIGHT=40, FLOW_K=1.3)     # cylinder_test settings
IRON = dict(
    CLEAR=3.0,       # face tracking: how far outside the face the nozzle tracks, mm
    TEMP=190,        # ironing nozzle temperature, C (soft but not dragging)   TODO
)

## Print the column (one arm)

In [ ]:
g, _ = cyl.parse([f"{k}={v}" for k, v in COLUMN.items()])
k = cyl.plan(g)
cyl.show(g, k)
# cell = Cell(g, SIM); cyl.run(g, k, cell)        # uncomment to print; same as: python3 cylinder_test.py T=0 R=40 E=4 ...

## Face tracking, cold and clear
The ironing arm goes to the column's radius plus `CLEAR`, at mid height, and holds there while the bed turns a revolution.
For a cylinder the point is fixed in the arm's frame (the face is a circle about the axis); for a rounded square it would be
`radius(a, θ) + CLEAR` updated every tick, exactly as in notebook 01. Watch the gap stay constant all the way round.

In [ ]:
from cell.runs.cylinder import DEFAULTS
g2 = {"arms": {IRON_ARM: dict(DEFAULTS[IRON_ARM], T=IRON_ARM, side=SIDE[IRON_ARM])}}
cell = Cell(g2, SIM)
ia = g2["arms"][IRON_ARM]
R_FACE = COLUMN["R"] + ia.get("DR", 0.0)               # the face this arm sees (its outline offset included)

def pt(a, r):
    ang = math.radians(a["ANGLE"]); return (a["CX"] + r * math.cos(ang), a["CY"] + r * math.sin(ang))

def iron_move(r, z, speed=10, wait=True):
    if z < ia["Z0"] - 1e-6: raise RuntimeError("below Z0")
    x, y = pt(ia, r); cell.move(IRON_ARM, x, y, z, speed=speed, wait=wait)

z_mid = ia["Z0"] + COLUMN["HEIGHT"] / 2                  # track at mid height
cell.move(IRON_ARM, *PARK, speed=60, wait=True)
cell.move(IRON_ARM, PARK[0], PARK[1], SAFE_Z, speed=60, wait=True)
x, y = pt(ia, R_FACE + 20); cell.move(IRON_ARM, x, y, SAFE_Z, speed=60, wait=True)
iron_move(R_FACE + 20, z_mid, speed=20)
iron_move(R_FACE + IRON["CLEAR"], z_mid, speed=5)       # creep in to the clearance
cell.tt.rotate_relative(360, 20, wait=True)             # one revolution: the gap should not change
iron_move(R_FACE + 20, z_mid, speed=10)
print("tracking pass done; measured gap constant? note it here:")

## Ironing pass: Nino's zones and motifs
Nino's method: the face is split into **zones**, each with a range along the wall, a start height, a pressure, a list of
passes (z offsets, swept back and forth) and a **motif**: `line`, `zigzag` or `spiral`. For each zone the nozzle comes in
front of it a safe distance away, approaches the face straight on until it presses by `pressure`, does the passes, and
backs away before the next zone. The pattern functions are hers (`src/cell/ironing.py`).

`FACE` says what the nozzle is pressing on. A **square column's side is flat**, so Nino's wall applies as she wrote it: a
straight line along the side at `apothem - pressure` from the axis, the arm moving in its own XYZ with the bed still
(`FLAT_HALF_WIDTH` = half side minus the fillet radius keeps zones off the fillets). A **cylinder's face is round**, so
"along the wall" is an arc about the axis; `IRON_MODE` then picks who makes it: `"arm"` sweeps the arc in XYZ with the bed
still, `"bed"` turns the face past a nozzle that holds its radius and does only the z part of the motif. Same zones, same
motifs in every case. Everything compiles to a speed-and-time program, previewed before anything moves. The nozzle is not tilted here (the cell's tool orientation is
fixed per arm); Nino's `IRON_A` tilt would be a tool-orientation change in cell studio if we want it.

Units: `x_start`/`x_end` in mm along the face, `z_start` in mm above the disc, `pressure` in mm into the face
(negative = clear of it, useful for a dry run).

In [ ]:
from cell import ironing as ir, program as prg

FACE = "round"            # "round" for the cylinder column above; "flat" for a square column's side (then R_FACE = half side + DR)
FLAT_HALF_WIDTH = None    # flat faces: half side minus the fillet radius, so zones stay off the fillets
IRON_MODE = "arm"         # "arm": bed still, the arm moves along the face (Nino's way); "bed": round faces only, the bed turns the face past the nozzle
IRON_SPEED = 5.0          # mm/s along the pattern
SAFE_DISTANCE = 5.0       # mm clear of the face between zones
Z_TOP = COLUMN["HEIGHT"]  # top of the wall, for the checks

# One dict per zone (Nino's format). Examples of each motif:
#   dict(pressure=0.5, x_start=-40, x_end=40, z_start=80, passes=range(0, 21, 10), motif="line",   step=1),
#   dict(pressure=0.5, x_start=-40, x_end=40, z_start=80, passes=range(0, 21, 10), motif="zigzag", amp=10, pitch=0.5),
#   dict(pressure=0.5, x_start=-40, x_end=40, z_start=80, passes=range(0, 21, 10), motif="spiral", r=1, pitch=1.8, seg=20),
IRON_ZONES = [
    dict(pressure=-2.2, x_start=-35, x_end=-25, z_start=10, passes=range(0, 21, 8), motif="zigzag", amp=10, pitch=0.5),   # dry: 2.2 mm clear
    dict(pressure=-2.0, x_start=-20, x_end=-10, z_start=10, passes=range(0, 21, 8), motif="zigzag", amp=10, pitch=0.5),
    # dict(pressure=0.2, x_start=5, x_end=25, z_start=10, passes=range(0, 21, 10), motif="spiral", r=1, pitch=1.8, seg=20),
]
for note in ir.check_zones(IRON_ZONES, COLUMN["L_H"], Z_TOP):
    print("warning:", note)

# look at the motifs in the (along the face, z) plane before compiling
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 4))
for k, cfg in enumerate(IRON_ZONES, start=1):
    pts = ir.zone_points(cfg)
    ax.plot([s for s, _ in pts], [z for _, z in pts], lw=0.8, label=f"zone {k}: {cfg['motif']}, press {cfg['pressure']:g}")
ax.set_xlabel("along the face, mm"); ax.set_ylabel("z above the disc, mm"); ax.set_aspect("equal"); ax.legend(); ax.grid(alpha=.3)
plt.show()

Compile the zones to a program and preview it: the bed track is what turns the face past the nozzle, the arm track is the
radius (pressure) and z. Then run it with the player, exactly as the panel does.

In [ ]:
iron_prog = ir.compile_zones(ia, R_FACE, IRON_ZONES, face=FACE, mode=IRON_MODE, flat_half_width=FLAT_HALF_WIDTH, iron_speed=IRON_SPEED, safe_distance=SAFE_DISTANCE, name="ironing")
print(iron_prog.summary())
rows = prg.simulate(iron_prog, tick=0.1)
import math
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r["t"] for r in rows], [r["bed_dps"] for r in rows]); ax[0].set_title("bed speed, deg/s (zero in arm mode)"); ax[0].set_xlabel("s")
rr = [math.hypot(r["arm1"][0] - ia["CX"], r["arm1"][1] - ia["CY"]) for r in rows if r["arm1"]]
zz = [r["arm1"][2] - ia["Z0"] for r in rows if r["arm1"]]
ax[1].plot([r["t"] for r in rows if r["arm1"]], rr, label="nozzle radius"); ax[1].plot([r["t"] for r in rows if r["arm1"]], zz, label="z above disc")
ax[1].axhline(R_FACE, ls="--", lw=0.8, color="k", label="face"); ax[1].legend(); ax[1].set_xlabel("s"); ax[1].set_title("ironing arm")
plt.tight_layout(); plt.show()

In [ ]:
if not SIM:
    cell.ext.heat_and_wait(IRON_ARM, IRON["TEMP"])
cell.gcode("EXTRUDE_OFF")
player = prg.Player(cell, iron_prog, z_floor={f"arm{IRON_ARM}": ia["Z0"]}, on_log=print)
player.start()
while player.running():
    st = player.status(); print(f"\r{st['t']:6.1f} / {st['total']:.1f} s   bed {cell.tt.get_angle():7.1f} deg", end="")
    time.sleep(1)
print("\nironing done; error:", player.error)
# player.stop()   # abort at any time: extruders off, bed stopped

## Opposite faces while printing
The printing arm and the ironing arm are 180° apart, so while the column is being laid on one side the other side is a few layers
old and can be ironed. The questions to settle in trials: how many layers behind the print head the ironing band should sit
(`LAG_LAYERS`), and the back-off rule when the printing arm has to cross. Skeleton below; it reuses the zone program for the ironing arm and
`cylinder_test.run` for the printing arm in a thread. **Do not run until the column, the face tracking and the ironing pass are each proven on their own.**

In [ ]:
LAG_LAYERS = 10        # TODO: layers between the print head and the ironing band
# plan:
#   1. start cyl.run(g, k, cell_print) in a thread (printing arm, bed master as usual)
#   2. the bed speed is set by the print, so the zones cannot drive the bed: compile them with the
#      bed track dropped and let the ironing arm follow z_print - LAG_LAYERS * L_H plus the motif z,
#      pressing at R_FACE - pressure, extruder off, hot
#   3. the pattern along the face then comes from the print's own rotation
#   4. stop ironing when the print's final flat loop starts; back off; let the print finish
print("skeleton only")

## What to measure
- Surface: photo under raking light, ironed vs unironed faces; profile gauge if available.
- Wall thickness at the ironed and unironed faces (calipers), to see how much material the press displaces.
- Any drag marks or tearing → TEMP too high or PRESS too deep.
- Note PRESS, TEMP, BED_DPS, Z_RATE for every trial in the table below.

| trial | PRESS | TEMP | BED_DPS | Z_RATE | result |
|---|---|---|---|---|---|
| 1 | | | | | |